# Santa Fe Laser — Full Pipeline (Colab Pro)

Self-contained notebook for the Deep Learning assignment (one-step training + 200-step recursive forecast on the Santa Fe laser series).

**What it does, end to end:**

1. Loads `Xtrain.mat`.
2. Builds five candidate architectures: MLP, LSTM, GRU, TCN, N-BEATS-lite.
3. Trains every architecture with noise injection, scheduled-sampling rollout loss, and EMA weights.
4. **Phase 1 — lookback search.** Internal 200-step holdout = last 200 points of `Xtrain`. Sweeps lookbacks for every architecture and picks the best per-architecture `(lookback, hyperparams)`.
5. **Phase 2 — multi-seed ensemble.** Trains 5 seeds per architecture with the chosen lookback and averages the 200-step recursive forecasts.
6. Reports MAE/MSE on the internal holdout, plots predictions vs. real, and tells you which architecture wins.
7. Saves the winning weights and predictions so that on **May 8** you only need to drop `Xtest.mat` into the notebook and run the final cell.

**Strictly inside the assignment:** the test set is *never* touched during training or model selection. Selection is done exclusively on the last 200 points of `Xtrain` (the recursive holdout). Lookback tuning is explicitly required by part (b).

## 1. Setup

In Colab, `torch`, `numpy`, `scipy`, and `matplotlib` are preinstalled. We only need `joblib`. Switch to a GPU runtime (Runtime → Change runtime type → T4/A100) before running.

In [ ]:
import sys, subprocess
def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pkgs])
try:
    import joblib  # noqa: F401
except ImportError:
    _pip_install(["joblib>=1.3"])
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
from __future__ import annotations
import copy, json, math, os, random, time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Sequence

import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR = Path("./santa_fe_work"); WORK_DIR.mkdir(exist_ok=True)
print("Working dir:", WORK_DIR.resolve(), "| device:", DEVICE)

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

## 2. Load `Xtrain.mat`

Run the cell below. If `Xtrain.mat` is not next to the notebook, an upload widget will appear (Colab). You can also drag the file into the file panel as `Xtrain.mat`.

In [ ]:
XTRAIN_PATH = Path("Xtrain.mat")
if not XTRAIN_PATH.exists():
    try:
        from google.colab import files  # type: ignore
        print("Upload Xtrain.mat ...")
        uploaded = files.upload()
        for name in uploaded:
            if name.endswith(".mat"):
                Path(name).rename("Xtrain.mat")
    except ImportError:
        raise FileNotFoundError("Place Xtrain.mat next to this notebook.")
assert XTRAIN_PATH.exists(), "Xtrain.mat is missing."

def load_mat_series(path: str | Path, key: str) -> np.ndarray:
    data = sio.loadmat(Path(path).as_posix())
    if key not in data:
        keys = [k for k in data if not k.startswith("__")]
        if not keys:
            raise ValueError(f"No data arrays in {path}.")
        key = keys[0]
    return np.asarray(data[key]).reshape(-1).astype(np.float32)

series = load_mat_series(XTRAIN_PATH, key="Xtrain")
print(f"Series length: {len(series)} | min={series.min():.2f} max={series.max():.2f} mean={series.mean():.2f}")

plt.figure(figsize=(10, 2.5))
plt.plot(series, lw=0.7)
plt.title("Santa Fe laser — Xtrain"); plt.xlabel("t"); plt.ylabel("intensity")
plt.tight_layout(); plt.show()

## 3. Data utilities

Min-max scaling to `[-1, 1]` (fit only on training portion — the last 200 points are held out and never used to fit the scaler), sliding-window construction, and split.

In [ ]:
HOLDOUT = 200

class MinMaxScaler11:
    def __init__(self):
        self.lo = None; self.hi = None
    def fit(self, x):
        x = np.asarray(x, np.float32).reshape(-1)
        self.lo = float(x.min()); self.hi = float(x.max()); return self
    def transform(self, x):
        x = np.asarray(x, np.float32).reshape(-1)
        d = max(self.hi - self.lo, 1e-8)
        return ((x - self.lo) / d * 2.0 - 1.0).astype(np.float32)
    def inverse(self, x):
        x = np.asarray(x, np.float32).reshape(-1)
        return ((x + 1.0) / 2.0 * (self.hi - self.lo) + self.lo).astype(np.float32)

def make_windows(series, lookback, horizon=1):
    series = np.asarray(series, np.float32).reshape(-1)
    n = len(series) - lookback - horizon + 1
    if n <= 0:
        raise ValueError("Series too short.")
    x = np.zeros((n, lookback), np.float32)
    y = np.zeros((n, horizon), np.float32)
    for i in range(n):
        x[i] = series[i:i + lookback]
        y[i] = series[i + lookback:i + lookback + horizon]
    return x, y

train_part = series[:-HOLDOUT]
holdout_part = series[-HOLDOUT:]
scaler = MinMaxScaler11().fit(train_part)
train_scaled = scaler.transform(train_part)
print(f"Train: {len(train_part)} samples | Holdout: {len(holdout_part)} samples (last 200)")

## 4. Models

Five candidate one-step-ahead regressors. All take a window of length `lookback` and output a single next-value prediction.

In [ ]:
def _seq(x):
    if x.ndim == 2: return x.unsqueeze(-1)
    return x

class MLP(nn.Module):
    def __init__(self, lookback, hidden=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(lookback, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        if x.ndim == 3: x = x.squeeze(-1)
        return self.net(x).squeeze(-1)

class LSTMReg(nn.Module):
    def __init__(self, lookback, hidden=64, num_layers=1, dropout=0.1):
        super().__init__()
        d = 0.0 if num_layers == 1 else dropout
        self.rnn = nn.LSTM(1, hidden, num_layers=num_layers, batch_first=True, dropout=d)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        x = _seq(x); y, _ = self.rnn(x)
        return self.head(y[:, -1, :]).squeeze(-1)

class GRUReg(nn.Module):
    def __init__(self, lookback, hidden=64, num_layers=1, dropout=0.1):
        super().__init__()
        d = 0.0 if num_layers == 1 else dropout
        self.rnn = nn.GRU(1, hidden, num_layers=num_layers, batch_first=True, dropout=d)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        x = _seq(x); y, _ = self.rnn(x)
        return self.head(y[:, -1, :]).squeeze(-1)

class _Chomp(nn.Module):
    def __init__(self, c): super().__init__(); self.c = c
    def forward(self, x): return x if self.c == 0 else x[:, :, :-self.c]

class _TBlock(nn.Module):
    def __init__(self, ci, co, k, dil, dropout):
        super().__init__()
        pad = (k - 1) * dil
        self.net = nn.Sequential(
            nn.Conv1d(ci, co, k, padding=pad, dilation=dil), _Chomp(pad), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(co, co, k, padding=pad, dilation=dil), _Chomp(pad), nn.ReLU(), nn.Dropout(dropout),
        )
        self.down = nn.Conv1d(ci, co, 1) if ci != co else None
        self.act = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.down is None else self.down(x)
        return self.act(out + res)

class TCN(nn.Module):
    def __init__(self, lookback, channels=64, num_blocks=4, kernel=3, dropout=0.1):
        super().__init__()
        layers = []
        ci = 1
        for i in range(num_blocks):
            layers.append(_TBlock(ci, channels, kernel, 2 ** i, dropout)); ci = channels
        self.tcn = nn.Sequential(*layers)
        self.head = nn.Linear(ci, 1)
    def forward(self, x):
        x = _seq(x).transpose(1, 2)
        h = self.tcn(x)
        return self.head(h[:, :, -1]).squeeze(-1)

class _NBBlock(nn.Module):
    def __init__(self, lookback, width, depth):
        super().__init__()
        layers = []; in_dim = lookback
        for _ in range(depth):
            layers += [nn.Linear(in_dim, width), nn.ReLU()]; in_dim = width
        self.body = nn.Sequential(*layers)
        self.bc = nn.Linear(width, lookback)
        self.fc = nn.Linear(width, 1)
    def forward(self, x):
        h = self.body(x); return self.bc(h), self.fc(h)

class NBeatsLite(nn.Module):
    def __init__(self, lookback, num_blocks=3, width=128, depth=4):
        super().__init__()
        self.blocks = nn.ModuleList([_NBBlock(lookback, width, depth) for _ in range(num_blocks)])
    def forward(self, x):
        if x.ndim == 3: x = x.squeeze(-1)
        residual = x
        forecast = torch.zeros((x.size(0), 1), dtype=x.dtype, device=x.device)
        for b in self.blocks:
            bc, fc = b(residual); residual = residual - bc; forecast = forecast + fc
        return forecast.squeeze(-1)

def build_model(name, lookback, hp):
    name = name.lower()
    if name == "mlp":    return MLP(lookback, hidden=hp["hidden"], dropout=hp["dropout"])
    if name == "lstm":   return LSTMReg(lookback, hidden=hp["hidden"], num_layers=hp["layers"], dropout=hp["dropout"])
    if name == "gru":    return GRUReg(lookback, hidden=hp["hidden"], num_layers=hp["layers"], dropout=hp["dropout"])
    if name == "tcn":    return TCN(lookback, channels=hp["hidden"], num_blocks=hp["layers"], kernel=3, dropout=hp["dropout"])
    if name == "nbeats": return NBeatsLite(lookback, num_blocks=hp["layers"], width=hp["hidden"], depth=hp.get("depth", 4))
    raise ValueError(name)

## 5. Training utilities

* **One-step MSE** as the primary loss.
* **Scheduled-sampling rollout loss** as an auxiliary term — feeds the model's own predictions back during training so it sees its own error distribution (helps recursive forecasting).
* **Gaussian noise injection** on inputs (regularization).
* **EMA (exponential moving average) weights** — usually noticeably better in recursive rollouts than the raw model.
* **Recursive 200-step holdout MAE** as the early-stopping signal — directly aligned with what the assignment evaluates on.

In [ ]:
@dataclass
class TrainCfg:
    name: str = "tcn"
    lookback: int = 20
    epochs: int = 250
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-4
    dropout: float = 0.1
    hidden: int = 64
    layers: int = 4
    noise_std: float = 0.02
    rollout_horizon: int = 8
    aux_rollout_weight: float = 0.2
    sched_sample_max: float = 0.5
    ema_decay: float = 0.999
    patience: int = 40
    seed: int = 42

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    def update(self, model):
        d = self.decay
        with torch.no_grad():
            for ep, mp in zip(self.shadow.parameters(), model.parameters()):
                ep.data.mul_(d).add_(mp.data, alpha=1.0 - d)
            for eb, mb in zip(self.shadow.buffers(), model.buffers()):
                eb.data.copy_(mb.data)
    def model(self):
        return self.shadow

def recursive_forecast_scaled(model, history_scaled, steps, lookback, device):
    model.eval()
    window = torch.from_numpy(history_scaled[-lookback:].astype(np.float32)).to(device)
    preds = []
    with torch.no_grad():
        for _ in range(steps):
            x = window[-lookback:].view(1, lookback)
            y = model(x).item()
            preds.append(float(y))
            window = torch.cat([window, torch.tensor([y], device=device)])
    return np.asarray(preds, np.float32)

def metrics(pred, true):
    pred = np.asarray(pred, np.float32).reshape(-1)
    true = np.asarray(true, np.float32).reshape(-1)
    return {"mae": float(np.mean(np.abs(pred - true))),
            "mse": float(np.mean((pred - true) ** 2))}

def rollout_loss(model, x_batch, y_batch, sched_prob):
    bsz, lb = x_batch.shape; H = y_batch.shape[1]
    window = x_batch; preds = []
    for t in range(H):
        p = model(window); preds.append(p)
        if t < H - 1:
            gt = y_batch[:, t]
            mix_mask = (torch.rand(bsz, device=x_batch.device) < sched_prob).float()
            mixed = mix_mask * p.detach() + (1.0 - mix_mask) * gt
            window = torch.cat([window[:, 1:], mixed.unsqueeze(-1)], dim=1)
    return nn.functional.mse_loss(torch.stack(preds, dim=1), y_batch)

def train_one(cfg: TrainCfg, train_scaled, holdout_real, scaler, device, verbose=False):
    set_seed(cfg.seed)
    x, y = make_windows(train_scaled, cfg.lookback, horizon=cfg.rollout_horizon)
    ds = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False)

    model = build_model(cfg.name, cfg.lookback,
                        {"hidden": cfg.hidden, "layers": cfg.layers, "dropout": cfg.dropout}).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    crit = nn.MSELoss()
    ema = EMA(model, cfg.ema_decay)

    best = {"mae": float("inf"), "mse": float("inf")}
    best_state = None
    patience_left = cfg.patience

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        p_sched = (epoch / cfg.epochs) * cfg.sched_sample_max
        for xb, yb in loader:
            xb = xb.to(device).float(); yb = yb.to(device).float()
            if cfg.noise_std > 0:
                xb = xb + torch.randn_like(xb) * cfg.noise_std
            pred = model(xb)
            loss = crit(pred, yb[:, 0])
            if cfg.rollout_horizon > 1 and cfg.aux_rollout_weight > 0:
                loss = loss + cfg.aux_rollout_weight * rollout_loss(model, xb, yb, p_sched)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            ema.update(model)
        ema_pred_scaled = recursive_forecast_scaled(ema.model(), train_scaled, len(holdout_real), cfg.lookback, device)
        ema_pred = scaler.inverse(ema_pred_scaled)
        m = metrics(ema_pred, holdout_real)
        if m["mae"] < best["mae"]:
            best = m; patience_left = cfg.patience
            best_state = copy.deepcopy(ema.model().state_dict())
        else:
            patience_left -= 1
            if patience_left <= 0:
                if verbose: print(f"  early stop @ epoch {epoch}")
                break
        if verbose and epoch % 25 == 0:
            print(f"  epoch {epoch:3d} | holdout MAE {m['mae']:.3f} | best {best['mae']:.3f}")
    return best, best_state

## 6. Phase 1 — lookback search per architecture

For each architecture we sweep a small grid of lookbacks (and one architecture-friendly default for hidden/layers). Selection metric: 200-step recursive MAE on the internal holdout. Single seed, 150 epochs — purely for picking `(arch, lookback)`.

If you're tight on Colab time, drop entries from `LOOKBACK_GRID` or `ARCH_DEFAULTS`.

In [ ]:
LOOKBACK_GRID = [10, 20, 30, 50]
ARCH_DEFAULTS = {
    "mlp":    {"hidden": 128, "layers": 2, "dropout": 0.1, "lr": 1e-3},
    "lstm":   {"hidden": 64,  "layers": 2, "dropout": 0.1, "lr": 1e-3},
    "gru":    {"hidden": 64,  "layers": 2, "dropout": 0.1, "lr": 1e-3},
    "tcn":    {"hidden": 64,  "layers": 4, "dropout": 0.1, "lr": 1e-3},
    "nbeats": {"hidden": 128, "layers": 3, "dropout": 0.0, "lr": 1e-3},
}
SEARCH_EPOCHS = 150
SEARCH_PATIENCE = 25

search_results = []
t0 = time.time()
for arch, defaults in ARCH_DEFAULTS.items():
    for lb in LOOKBACK_GRID:
        cfg = TrainCfg(
            name=arch, lookback=lb, epochs=SEARCH_EPOCHS, batch_size=64,
            lr=defaults["lr"], weight_decay=1e-4, dropout=defaults["dropout"],
            hidden=defaults["hidden"], layers=defaults["layers"],
            noise_std=0.02, rollout_horizon=8, aux_rollout_weight=0.2,
            sched_sample_max=0.5, ema_decay=0.999, patience=SEARCH_PATIENCE, seed=1234,
        )
        ts = time.time()
        best, _ = train_one(cfg, train_scaled, holdout_part, scaler, DEVICE, verbose=False)
        dt = time.time() - ts
        search_results.append({"arch": arch, "lookback": lb, **best, "secs": round(dt, 1)})
        print(f"{arch:6s} lb={lb:>3d}  MAE={best['mae']:8.3f}  MSE={best['mse']:10.2f}  ({dt:5.1f}s)")
print(f"\nPhase 1 total time: {time.time() - t0:.1f}s")
(WORK_DIR / "phase1_search.json").write_text(json.dumps(search_results, indent=2))

In [ ]:
best_per_arch = {}
for r in search_results:
    a = r["arch"]
    if a not in best_per_arch or r["mae"] < best_per_arch[a]["mae"]:
        best_per_arch[a] = r
print("Best lookback per architecture (by 200-step recursive MAE on holdout):")
for a, r in sorted(best_per_arch.items(), key=lambda kv: kv[1]["mae"]):
    print(f"  {a:6s} -> lookback={r['lookback']:>3d}  MAE={r['mae']:8.3f}  MSE={r['mse']:10.2f}")

## 7. Phase 2 — multi-seed ensemble per architecture

For each architecture's best lookback, train 5 seeds (longer, 250 epochs) and average the recursive predictions. The mean reduces seed-to-seed variance — strictly an averaging trick on top of the assignment, no extra data.

In [ ]:
N_SEEDS = 5
FINAL_EPOCHS = 250
FINAL_PATIENCE = 40

ensemble_results = {}
ensemble_states = {}

t0 = time.time()
for arch, defaults in ARCH_DEFAULTS.items():
    lb = best_per_arch[arch]["lookback"]
    print(f"\n=== {arch} (lookback={lb}) ===")
    seed_metrics = []; seed_preds = []; seed_states = []
    for s in range(N_SEEDS):
        cfg = TrainCfg(
            name=arch, lookback=lb, epochs=FINAL_EPOCHS, batch_size=64,
            lr=defaults["lr"], weight_decay=1e-4, dropout=defaults["dropout"],
            hidden=defaults["hidden"], layers=defaults["layers"],
            noise_std=0.02, rollout_horizon=8, aux_rollout_weight=0.2,
            sched_sample_max=0.5, ema_decay=0.999, patience=FINAL_PATIENCE, seed=2000 + s,
        )
        best, state = train_one(cfg, train_scaled, holdout_part, scaler, DEVICE, verbose=False)
        seed_metrics.append(best); seed_states.append(state)
        m_tmp = build_model(arch, lb, {"hidden": defaults["hidden"], "layers": defaults["layers"], "dropout": defaults["dropout"]}).to(DEVICE)
        m_tmp.load_state_dict(state); m_tmp.eval()
        pred_scaled = recursive_forecast_scaled(m_tmp, train_scaled, len(holdout_part), lb, DEVICE)
        seed_preds.append(scaler.inverse(pred_scaled))
        print(f"  seed {s} -> MAE={best['mae']:8.3f}  MSE={best['mse']:10.2f}")
    preds_arr = np.stack(seed_preds, axis=0)
    ens_pred = preds_arr.mean(axis=0)
    ens_metrics = metrics(ens_pred, holdout_part)
    print(f"  ENSEMBLE  -> MAE={ens_metrics['mae']:8.3f}  MSE={ens_metrics['mse']:10.2f}")
    ensemble_results[arch] = {
        "lookback": lb, "seed_metrics": seed_metrics,
        "ensemble_metrics": ens_metrics, "preds": preds_arr,
        "ens_pred": ens_pred,
    }
    ensemble_states[arch] = seed_states
print(f"\nPhase 2 total time: {time.time() - t0:.1f}s")

## 8. Comparison table & plots

`ensemble_mae` is the metric that matters — it's exactly what the assignment will be graded on, just computed on the internal holdout instead of the released `Xtest`.

In [ ]:
rows = []
for arch, info in ensemble_results.items():
    seed_maes = [m["mae"] for m in info["seed_metrics"]]
    rows.append({
        "arch": arch,
        "lookback": info["lookback"],
        "ensemble_mae": info["ensemble_metrics"]["mae"],
        "ensemble_mse": info["ensemble_metrics"]["mse"],
        "seed_mae_mean": float(np.mean(seed_maes)),
        "seed_mae_std": float(np.std(seed_maes)),
    })
rows.sort(key=lambda r: r["ensemble_mae"])

print(f"{'arch':6s} {'lb':>3s} {'ens_MAE':>10s} {'ens_MSE':>12s} {'seedMAE_mean':>14s} {'seedMAE_std':>13s}")
for r in rows:
    print(f"{r['arch']:6s} {r['lookback']:>3d} {r['ensemble_mae']:>10.3f} {r['ensemble_mse']:>12.2f} {r['seed_mae_mean']:>14.3f} {r['seed_mae_std']:>13.3f}")

best_arch = rows[0]["arch"]
print(f"\n>>> BEST architecture on internal holdout: {best_arch} (lookback={rows[0]['lookback']}, ensemble MAE={rows[0]['ensemble_mae']:.3f})")

plt.figure(figsize=(11, 4))
plt.plot(holdout_part, label="real", color="black", lw=1.2)
for arch, info in ensemble_results.items():
    plt.plot(info["ens_pred"], label=f"{arch} (MAE {info['ensemble_metrics']['mae']:.2f})", lw=0.9)
plt.title("Internal 200-step recursive holdout — ensemble predictions")
plt.xlabel("step"); plt.ylabel("intensity"); plt.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.savefig(WORK_DIR / "compare_holdout.pdf"); plt.show()

## 9. Refit best ensemble on full `Xtrain` and save

Once we've chosen the architecture & lookback on the internal holdout, we refit the same `(arch, lookback, hyperparams)` on the **entire** `Xtrain.mat` (no holdout — the test set is what's left over). Same 5 seeds, same training recipe.

This gives the ensemble more data to fit, which is the standard final-model step. The 200-step horizon for `Xtest` is then produced recursively from the last `lookback` points of `Xtrain`.

In [ ]:
FINAL_ARCH = best_arch
FINAL_LOOKBACK = best_per_arch[FINAL_ARCH]["lookback"]
FINAL_HP = ARCH_DEFAULTS[FINAL_ARCH]

scaler_full = MinMaxScaler11().fit(series)
full_scaled = scaler_full.transform(series)

def train_one_full(cfg: TrainCfg, full_scaled, device):
    """Same as train_one but with no holdout — train for fixed epochs and keep EMA."""
    set_seed(cfg.seed)
    x, y = make_windows(full_scaled, cfg.lookback, horizon=cfg.rollout_horizon)
    ds = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False)

    model = build_model(cfg.name, cfg.lookback,
                        {"hidden": cfg.hidden, "layers": cfg.layers, "dropout": cfg.dropout}).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    crit = nn.MSELoss(); ema = EMA(model, cfg.ema_decay)
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        p_sched = (epoch / cfg.epochs) * cfg.sched_sample_max
        for xb, yb in loader:
            xb = xb.to(device).float(); yb = yb.to(device).float()
            if cfg.noise_std > 0: xb = xb + torch.randn_like(xb) * cfg.noise_std
            pred = model(xb); loss = crit(pred, yb[:, 0])
            if cfg.rollout_horizon > 1 and cfg.aux_rollout_weight > 0:
                loss = loss + cfg.aux_rollout_weight * rollout_loss(model, xb, yb, p_sched)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            ema.update(model)
    return copy.deepcopy(ema.model().state_dict())

print(f"Refitting {FINAL_ARCH} on full Xtrain (lookback={FINAL_LOOKBACK})")
final_states = []
final_preds_holdout = []
for s in range(N_SEEDS):
    cfg = TrainCfg(
        name=FINAL_ARCH, lookback=FINAL_LOOKBACK, epochs=FINAL_EPOCHS, batch_size=64,
        lr=FINAL_HP["lr"], weight_decay=1e-4, dropout=FINAL_HP["dropout"],
        hidden=FINAL_HP["hidden"], layers=FINAL_HP["layers"],
        noise_std=0.02, rollout_horizon=8, aux_rollout_weight=0.2,
        sched_sample_max=0.5, ema_decay=0.999, patience=FINAL_PATIENCE, seed=3000 + s,
    )
    state = train_one_full(cfg, full_scaled, DEVICE)
    final_states.append(state)
    print(f"  full-fit seed {s} done")

final_meta = {
    "arch": FINAL_ARCH,
    "lookback": FINAL_LOOKBACK,
    "hp": FINAL_HP,
    "n_seeds": N_SEEDS,
    "scaler": {"lo": scaler_full.lo, "hi": scaler_full.hi},
    "internal_holdout_ensemble_mae": ensemble_results[FINAL_ARCH]["ensemble_metrics"]["mae"],
    "internal_holdout_ensemble_mse": ensemble_results[FINAL_ARCH]["ensemble_metrics"]["mse"],
}
torch.save({"states": final_states, "meta": final_meta}, WORK_DIR / "final_ensemble.pt")
(WORK_DIR / "final_meta.json").write_text(json.dumps(final_meta, indent=2))
print(f"\nSaved {WORK_DIR / 'final_ensemble.pt'}")

In [ ]:
def ensemble_recursive_forecast(states, arch, lookback, hp, history_scaled, steps, device):
    preds = []
    for st in states:
        m = build_model(arch, lookback, {"hidden": hp["hidden"], "layers": hp["layers"], "dropout": hp["dropout"]}).to(device)
        m.load_state_dict(st); m.eval()
        preds.append(recursive_forecast_scaled(m, history_scaled, steps, lookback, device))
    return np.stack(preds, axis=0).mean(axis=0)

ens_pred_scaled = ensemble_recursive_forecast(
    final_states, FINAL_ARCH, FINAL_LOOKBACK, FINAL_HP, full_scaled, steps=200, device=DEVICE,
)
ens_pred_real = scaler_full.inverse(ens_pred_scaled)
np.save(WORK_DIR / "predictions_xtest_ensemble.npy", ens_pred_real)
print("Saved 200-step recursive forecast to predictions_xtest_ensemble.npy")
print("First 5 predicted values:", ens_pred_real[:5])

plt.figure(figsize=(11, 3))
plt.plot(np.arange(len(series)), series, lw=0.6, label="Xtrain")
plt.plot(np.arange(len(series), len(series) + 200), ens_pred_real, lw=1.2, color="red", label="recursive forecast")
plt.title(f"Final ensemble forecast — {FINAL_ARCH}, lookback={FINAL_LOOKBACK}")
plt.xlabel("t"); plt.ylabel("intensity"); plt.legend()
plt.tight_layout(); plt.savefig(WORK_DIR / "final_forecast.pdf"); plt.show()

## 10. May 8 — final evaluation on `Xtest.mat`

When `Xtest.mat` is released, drop it next to the notebook (or upload it via the cell below) and run the next cell. It loads the ensemble we saved, regenerates the 200-step recursive forecast, and reports MAE/MSE + a comparison plot.

In [ ]:
XTEST_PATH = Path("Xtest.mat")
if not XTEST_PATH.exists():
    try:
        from google.colab import files  # type: ignore
        print("Upload Xtest.mat ...")
        uploaded = files.upload()
        for name in uploaded:
            if name.endswith(".mat"):
                Path(name).rename("Xtest.mat")
    except ImportError:
        print("Place Xtest.mat next to this notebook and rerun this cell.")

if XTEST_PATH.exists():
    test_series = load_mat_series(XTEST_PATH, key="Xtest")
    steps = len(test_series)
    print(f"Xtest length: {steps}")

    bundle = torch.load(WORK_DIR / "final_ensemble.pt", map_location=DEVICE, weights_only=False)
    meta = bundle["meta"]; states = bundle["states"]
    sc = MinMaxScaler11(); sc.lo = meta["scaler"]["lo"]; sc.hi = meta["scaler"]["hi"]
    full_scaled_eval = sc.transform(series)

    ens_pred_scaled = ensemble_recursive_forecast(
        states, meta["arch"], meta["lookback"], meta["hp"], full_scaled_eval, steps=steps, device=DEVICE,
    )
    ens_pred_real = sc.inverse(ens_pred_scaled)

    final_metrics = metrics(ens_pred_real, test_series)
    print(f"\nFINAL on Xtest -> MAE={final_metrics['mae']:.4f}  MSE={final_metrics['mse']:.4f}")

    np.save(WORK_DIR / "predictions_xtest_final.npy", ens_pred_real)
    (WORK_DIR / "final_metrics.json").write_text(json.dumps({
        **final_metrics, **meta, "steps": int(steps),
    }, indent=2))

    plt.figure(figsize=(11, 3.2))
    plt.plot(test_series, label="real Xtest", color="black", lw=1.0)
    plt.plot(ens_pred_real, label="ensemble forecast", color="red", lw=1.0)
    plt.title(f"Xtest — {meta['arch']} ensemble (lookback={meta['lookback']}) — MAE={final_metrics['mae']:.2f}, MSE={final_metrics['mse']:.2f}")
    plt.xlabel("step"); plt.ylabel("intensity"); plt.legend()
    plt.tight_layout(); plt.savefig(WORK_DIR / "final_pred_vs_real.pdf"); plt.show()